# Week 8 - Task 1: Evaluation Metrics for Imbalanced Datasets

## Objective

This notebook studies classification evaluation when the positive class is rare.

It demonstrates:
- a baseline classifier without imbalance handling,
- custom ROC and Precision-Recall curve calculations,
- threshold analysis,
- ROC-AUC and PR-AUC,
- macro, micro, and weighted F1 scores,
- and metric selection for a fraud-detection business context.

The dataset is synthetically generated with a deliberately severe class imbalance so the experiment is reproducible without an external download.

## Business Context

Assume the model is used to identify potentially fraudulent transactions.

Fraud is rare compared with legitimate transactions. In this context:
- Accuracy can look excellent even when the model misses many fraud cases.
- Recall is important when missing fraud is expensive.
- Precision matters because investigating false alarms costs time and money.
- F1 balances precision and recall.
- Precision-Recall curves are especially informative for rare positive classes.
- ROC-AUC remains useful for measuring ranking ability, but can appear optimistic under severe class imbalance.

The final metric should therefore be chosen based on the relative business cost of false negatives and false positives.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report
)

from src.custom_metrics import (
    custom_roc_curve,
    custom_precision_recall_curve,
    f1_scores_by_average,
    threshold_metrics
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
ASSET_DIR = "../docs/assets"
os.makedirs(ASSET_DIR, exist_ok=True)

In [ ]:
# Generate a highly imbalanced fraud-like dataset.
X, y = make_classification(
    n_samples=12000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_clusters_per_class=2,
    weights=[0.985, 0.015],
    class_sep=1.0,
    flip_y=0.002,
    random_state=RANDOM_STATE
)

X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
y = pd.Series(y, name="fraud")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print("Dataset shape:", X.shape)
print("Class distribution:")
display(y.value_counts().rename(index={0:"legitimate", 1:"fraud"}))
print("Fraud rate:", round(y.mean() * 100, 2), "%")

## 1. Baseline Classifier

The baseline Logistic Regression model is trained without class weighting or resampling.

This intentionally establishes the metric disparity caused by class imbalance.

In [ ]:
model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob),
    "PR_AUC": average_precision_score(y_test, y_prob)
}

display(pd.DataFrame([baseline_metrics]).round(4))
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

## 2. Custom ROC Curve

The ROC curve plots True Positive Rate against False Positive Rate for many thresholds.

The implementation below does not call `roc_curve`; it explicitly calculates the points by sorting prediction scores and evaluating thresholds.

In [ ]:
fpr, tpr, roc_thresholds = custom_roc_curve(y_test.to_numpy(), y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Custom ROC (AUC = {roc_auc_score(y_test, y_prob):.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title("Custom ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "roc_curve.png"), dpi=160)
plt.show()

## 3. Custom Precision-Recall Curve

The Precision-Recall curve is often more informative than ROC when the positive class is rare.

It focuses directly on:
- Precision: how many predicted positives are truly positive.
- Recall: how many actual positives are detected.

The baseline precision level is approximately the positive-class prevalence.

In [ ]:
precision_vals, recall_vals, pr_thresholds = custom_precision_recall_curve(
    y_test.to_numpy(), y_prob
)

plt.figure(figsize=(7, 6))
plt.plot(
    recall_vals,
    precision_vals,
    label=f"Custom PR (AP = {average_precision_score(y_test, y_prob):.4f})"
)
plt.axhline(
    y=y_test.mean(),
    linestyle="--",
    label=f"Positive prevalence = {y_test.mean():.3f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Custom Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "precision_recall_curve.png"), dpi=160)
plt.show()

## 4. Threshold Trade-offs

A probability threshold of 0.50 is not necessarily optimal.

For fraud detection, lowering the threshold can increase recall and catch more fraud, but it usually increases false positives and reduces precision.

The table below evaluates several thresholds.

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_table = threshold_metrics(y_test.to_numpy(), y_prob, thresholds)
display(threshold_table.round(4))

threshold_table.to_csv(
    os.path.join(ASSET_DIR, "threshold_analysis.csv"),
    index=False
)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(threshold_table["threshold"], threshold_table["precision"], label="Precision")
plt.plot(threshold_table["threshold"], threshold_table["recall"], label="Recall")
plt.plot(threshold_table["threshold"], threshold_table["f1"], label="F1")
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.title("Precision-Recall-F1 vs Threshold")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "threshold_tradeoff.png"), dpi=160)
plt.show()

## 5. Macro, Micro, and Weighted F1

For binary classification, the averaging method changes how class imbalance affects the reported F1 score.

- **Macro F1:** calculates F1 for each class and gives each class equal weight.
- **Micro F1:** aggregates all true positives, false positives, and false negatives before calculating the score; it is influenced by the larger class.
- **Weighted F1:** averages class-specific F1 scores weighted by class support.

For highly imbalanced data, macro F1 is useful when minority-class performance deserves equal importance.

In [ ]:
f1_comparison = f1_scores_by_average(y_test.to_numpy(), y_pred)

display(pd.DataFrame(
    list(f1_comparison.items()),
    columns=["Averaging", "F1"]
).round(4))

## 6. Business Metric Selection

For this simulated fraud-detection context, accuracy is not the primary metric because a model can obtain high accuracy by mostly predicting the majority legitimate class.

A practical monitoring set should include:
- **Recall:** to monitor missed fraud.
- **Precision:** to control investigation workload.
- **F1:** when precision and recall have similar business importance.
- **PR-AUC:** particularly useful for comparing ranking quality when fraud prevalence is low.
- **ROC-AUC:** useful as a broader ranking metric, but should not be considered alone.

If the cost of missing a fraudulent transaction is much higher than investigating a false positive, a lower threshold and higher-recall operating point may be preferable.

In [ ]:
summary = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob),
    "PR_AUC": average_precision_score(y_test, y_prob),
    "Positive_Prevalence": y_test.mean()
}])

display(summary.round(4))
summary.to_csv(os.path.join(ASSET_DIR, "evaluation_summary.csv"), index=False)
print("Evaluation assets saved to:", ASSET_DIR)